# Silver Weather

## 1. Business Purpose
Weather Forecast Silver dùng để cung cấp dữ liệu dự báo
thời tiết theo giờ tại từng warehouse của FastOrder.

Dữ liệu này có thể phục vụ:
- phân tích ảnh hưởng thời tiết tới orders / delivery
- logistics planning
- warehouse operations
- các mô hình analytics / forecasting sau này

## 2. Overview Data

In [0]:
import os

account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

endpoint = f"{account_name}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{endpoint}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{endpoint}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{endpoint}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{endpoint}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{endpoint}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

### Path

In [0]:
path_metadata = "weather/open_meteo/forecast/ingestion_date=2026-08-16/warehouse_id=WH_HCM/ingestion_id=9ece2b40-aacc-57e8-bc3f-cb6b85e8d233/metadata.json"
path_response = "weather/open_meteo/forecast/ingestion_date=2026-08-16/warehouse_id=WH_HCM/ingestion_id=9ece2b40-aacc-57e8-bc3f-cb6b85e8d233/response.json"

### Metadata

In [0]:
df_metadata_weather = spark.read.format("json").option("multiline", True).load(f"abfss://bronze@fastorderdatalake.dfs.core.windows.net/{path_metadata}")

df_metadata_weather.display()

In [0]:
df_metadata_weather.printSchema()

### response

In [0]:
from pyspark.sql import functions as F
df_response_weather = spark.read.format("json").option("multiline", True).load(f"abfss://bronze@fastorderdatalake.dfs.core.windows.net/{path_response}")

df_response_weather.display()

In [0]:
df_response_weather.printSchema()

## 3. Silver Design
- Grain 
- Key 
- Columns 
- Lineage 
- DQ rules 

## 3. Silver Design

### 3.1. Target Dataset

Dataset Silver đầu tiên cho Weather Forecast:

`weather_forecast_hourly`

Mục tiêu của dataset là chuyển các Forecast snapshot dạng JSON trong Bronze thành dữ liệu dạng bảng theo giờ, dễ dàng sử dụng cho analytics và các pipeline downstream.

---

### 3.2. Grain

Grain của dataset:

**1 row = 1 warehouse × 1 forecast snapshot × 1 forecast hour**

Ví dụ, cùng một warehouse và cùng một `forecast_time` có thể xuất hiện nhiều lần nếu chúng thuộc các Forecast snapshot khác nhau.

Điều này không được xem là duplicate vì mỗi snapshot thể hiện dự báo mà hệ thống nhận được tại một thời điểm khác nhau.

Ví dụ:

| warehouse_id | snapshot_at | forecast_time | temperature_2m |
|---|---|---|---:|
| WH_HCM | 2026-08-16 07:00 UTC | 2026-08-16 10:00 UTC | 30.2 |
| WH_HCM | 2026-08-16 08:00 UTC | 2026-08-16 10:00 UTC | 29.8 |

Hai record trên cùng dự báo cho `10:00`, nhưng thuộc hai Forecast snapshot khác nhau.

---

### 3.3. Candidate Key

Candidate key của dataset:

`warehouse_id + ingestion_id + forecast_time`

Trong đó:

- `warehouse_id`: warehouse mà Forecast được lấy cho.
- `ingestion_id`: định danh duy nhất của một Forecast ingestion snapshot.
- `forecast_time`: thời điểm mà giá trị thời tiết đang dự báo cho.

Candidate key này phải xác định duy nhất một row trong Silver.

---

### 3.4. Silver Columns

#### Forecast identity and context

| Column | Ý nghĩa |
|---|---|
| `warehouse_id` | Warehouse mà dữ liệu Forecast thuộc về |
| `ingestion_id` | Định danh Forecast ingestion snapshot |
| `snapshot_at` | Logical time của Airflow run tạo snapshot |
| `retrieved_at` | Thời điểm FastOrder thực sự gọi Open-Meteo |
| `forecast_time` | Thời điểm mà giá trị weather đang dự báo cho |

#### Weather measurements

| Column | Ý nghĩa |
|---|---|
| `temperature_2m` | Nhiệt độ không khí ở độ cao 2 m |
| `relative_humidity_2m` | Độ ẩm tương đối ở độ cao 2 m |
| `precipitation` | Lượng mưa |
| `wind_speed_10m` | Tốc độ gió ở độ cao 10 m |
| `weather_code` | Mã điều kiện thời tiết |

#### Location context

| Column | Ý nghĩa |
|---|---|
| `requested_latitude` | Latitude mà FastOrder gửi tới Open-Meteo |
| `requested_longitude` | Longitude mà FastOrder gửi tới Open-Meteo |
| `response_latitude` | Latitude của model/grid mà Open-Meteo thực tế sử dụng |
| `response_longitude` | Longitude của model/grid mà Open-Meteo thực tế sử dụng |

---

### 3.5. Time Standard

Tất cả timestamp trong Silver được chuẩn hóa về **UTC**.

- `snapshot_at`: UTC
- `retrieved_at`: UTC
- `forecast_time`: UTC

Open-Meteo hiện trả `hourly.time` theo timezone `Asia/Ho_Chi_Minh`, vì vậy `forecast_time` phải được chuyển từ giờ Việt Nam sang UTC trước khi ghi vào Silver.

Việc chuẩn hóa UTC giúp Weather có thể được kết hợp nhất quán với các nguồn dữ liệu khác như Orders, Clickstream và Delivery.

---

### 3.6. Bronze-to-Silver Relationship

Một Bronze Forecast ingestion gồm:

`metadata.json`

- chứa ingestion context và lineage như `warehouse_id`, `ingestion_id`, `logical_at`, `requested_at`, coordinates,...

`response.json`

- chứa Forecast payload từ Open-Meteo.
- trường `hourly` chứa các array thời gian và weather measurements.

`_SUCCESS`

- xác nhận ingestion unit đã commit thành công.

Một Bronze Forecast ingestion chứa nhiều forecast hours, do đó:

**1 committed Bronze ingestion → nhiều Silver hourly rows**

Với cấu hình Forecast hiện tại:

**1 ingestion → 48 forecast-hour rows**

## 4. Transformation
Chuyển các đơn vị nhập liệu Dự báo Thời tiết Bronze đã cam kết và chưa xử lý sang `weather_forecast_hourly` Silver Candidate.

Logic chuyển đổi trong môi trường sản xuất được triển khai ở:

`fastorder/transformation/silver/weather/forecast_hourly.py`

Notebook này đảm nhiệm việc điều phối quá trình chuyển đổi và kiểm tra kết quả trung gian.

In [0]:
import importlib
import fastorder.transformation.silver.weather.forecast_hourly as forecast_hourly

importlib.reload(forecast_hourly)

In [0]:
from fastorder.transformation.silver.weather.forecast_hourly import (
    discover_committed_forecast_ingestions,
    get_processed_forecast_ingestion_ids,
    find_pending_forecast_ingestions,
    load_pending_forecast_bronze,
    transform_forecast_hourly,
    profile_forecast_data_quality,
    assert_forecast_data_quality,
    profile_forecast_validation,
    assert_forecast_validation,
    write_forecast_silver,
    prepare_forecast_silver_output,
    extract_ingestion_id_from_path,
    run_forecast_silver_pipeline
)
from fastorder.storage.adls_client import get_adls_service_client

### 4.1. Load Pending Bronze Forecast Data

Xác định các đơn vị nhập liệu Dự báo đủ điều kiện cho xử lý Silver.

Một đơn vị nhập liệu phải thỏa cả hai điều kiện sau:

1. Nó đã được cam kết ở Bronze (`_SUCCESS` tồn tại).
2. `ingestion_id` của nó chưa được lưu lại ở Silver.

Các đơn vị nhập liệu đang chờ này là những dữ liệu Bronze duy nhất được xử lý trong lần chạy Silver hiện tại.

In [0]:
BRONZE_FORECAST_ROOT = "weather/open_meteo/forecast"

BRONZE_ABFSS_ROOT = (
    "abfss://bronze@fastorderdatalake.dfs.core.windows.net"
)
SILVER_FORECAST_PATH = (
    "abfss://silver@fastorderdatalake.dfs.core.windows.net/"
    "weather/forecast_hourly/"
)

In [0]:
service_client = get_adls_service_client()

bronze_client = service_client.get_file_system_client(
    "bronze"
)

#### 4.1.1. Discover Committed Bronze Ingestions

Quét đường dẫn Dự Báo Thời Tiết Bronze và xác định các đơn vị nhập dữ liệu có chứa dấu hiệu commit `_SUCCESS`.

Chỉ những đơn vị nhập dữ liệu đã commit thành công mới đủ điều kiện để xử lý tiếp ở Silver.

In [0]:
committed_ingestion_paths = (
    discover_committed_forecast_ingestions(
        bronze_client=bronze_client,
        forecast_root=BRONZE_FORECAST_ROOT,
    )
)

print(
    "Committed Bronze ingestions:",
    len(committed_ingestion_paths)
)

#### 4.1.2. Load Processed Silver Ingestion IDs

Đọc các giá trị `ingestion_id` riêng biệt đã được lưu trong bộ dữ liệu Delta Silver `weather_forecast_hourly`.

Những ID này đại diện cho các đơn vị ingestion Bronze đã hoàn tất xử lý Silver.

Nếu bộ dữ liệu Silver chưa tồn tại, tập hợp ingestion đã xử lý sẽ trống.

In [0]:
processed_ingestion_ids = (
    get_processed_forecast_ingestion_ids(
        spark=spark,
        silver_path=SILVER_FORECAST_PATH,
    )
)

print(
    "Processed Silver ingestions:",
    len(processed_ingestion_ids)
)

#### 4.1.3. Identify Pending Forecast Ingestions

So sánh các đơn vị hấp thụ Bronze đã cam kết với các ID hấp thụ đã được lưu trong Silver.

Về mặt khái niệm:

`Bronze đã cam kết - Silver đã xử lý = Các lần hấp thụ đang chờ`

Chỉ những đơn vị hấp thụ đang chờ mới tiếp tục cho lần biến đổi hiện tại.

In [0]:
pending_ingestion_paths = (
    find_pending_forecast_ingestions(
        committed_ingestion_paths,
        processed_ingestion_ids,
    )
)

print(
    "Pending Forecast ingestions:",
    len(pending_ingestion_paths)
)

#### 4.1.4. Bulk Load Pending Bronze Data

Tải `response.json` và `metadata.json` cho tất cả các đơn vị dự báo đang chờ xử lý vào DataFrame của Spark.

Các tệp đang chờ được đọc theo lô thay vì đọc và gộp từng đơn vị xử lý riêng lẻ.

In [0]:
if not pending_ingestion_paths:
    print(
        "No pending Forecast ingestions. "
        "Silver is already up to date."
    )

    df_response = None
    df_metadata = None
else:
    df_response, df_metadata = (
        load_pending_forecast_bronze(
            spark=spark,
            pending_ingestion_paths=pending_ingestion_paths,
            bronze_abfss_root=BRONZE_ABFSS_ROOT,
        )
    )

In [0]:
if df_response is not None:
    display(df_response.limit(5))
    df_response.printSchema()

In [0]:
if df_metadata is not None:
    display(df_metadata.limit(5))
    df_metadata.printSchema()

### 4.2. Transform Forecast Hourly Data

Áp dụng quy trình biến đổi Dự báo Thời tiết tái sử dụng được:

1. Lấy bối cảnh nhập dữ liệu từ đường dẫn nguồn Bronze.
2. Chuyển các mảng giờ của Open-Meteo thành các bản ghi theo giờ.
3. Gắn siêu dữ liệu nhập dữ liệu.
4. Chuẩn hóa cấu trúc Silver.
5. Chuẩn hóa dấu thời gian về UTC.

Kết quả của bước này là **Ứng viên Silver**.

Ứng viên Silver đã có cấu trúc Silver mục tiêu nhưng chưa được lưu lại.
Nó cần phải vượt qua kiểm tra Chất lượng Dữ liệu và Xác thực trước.

In [0]:
if df_response is not None:
    df_silver_candidate = (
        transform_forecast_hourly(
            df_response=df_response,
            df_metadata=df_metadata,
        )
    )

In [0]:
display(df_silver_candidate.limit(20))

In [0]:
df_silver_candidate.printSchema()

## 5. Data Quality

Đánh giá Ứng viên Silver trước khi cho phép nó tiến hành xác thực biến đổi cuối cùng.

Các kiểm tra Chất lượng Dữ liệu phát hiện các giá trị không hợp lệ nhưng không tự động sửa, bỏ hoặc chỉnh sửa bản ghi.

Các quy tắc hiện tại xác thực:

- các trường nhận dạng và dấu thời gian bắt buộc
- các đo lường thời tiết NULL
- phạm vi độ ẩm
- lượng mưa không âm
- tốc độ gió không âm
- tính duy nhất của hạt Silver

### 5.1. Data Quality Profiling

Đếm số bản ghi vi phạm từng quy tắc Chất lượng Dữ liệu.

Mỗi chỉ số theo quy ước:

- `0` → không có vi phạm
- `> 0` → có vấn đề về Chất lượng Dữ liệu

In [0]:
dq_result = profile_forecast_data_quality(
    df_silver_candidate
)

dq_result

### 5.2. Data Quality Assertion

Chặn xử lý Silver khi phát hiện một hoặc nhiều vi phạm quan trọng về Chất lượng Dữ liệu. Dữ liệu không hợp lệ sẽ không bị tự động chỉnh sửa hay lưu vào Silver.

In [0]:
assert_forecast_data_quality(
    dq_result
)

print("Forecast Data Quality PASS")

## 6. Validation

Xác nhận rằng quá trình chuyển đổi từ Bronze → Silver đã xử lý xong tất cả các đơn vị dự báo đang chờ.

Khác với Data Quality, chỉ kiểm tra giá trị của bản ghi, bước này kiểm tra tính đầy đủ của pipeline.

Các kiểm tra xác thực:

- mọi dữ liệu Bronze đang chờ đều xuất hiện trong Silver Candidate
- mỗi lần nhập dữ liệu tạo ra đúng 48 dòng dự báo giờ
- số đơn vị nhập đã xử lý khớp với kỳ vọng
- tổng số dòng trong Silver Candidate khớp với số dòng dự kiến

### 6.1. Transformation Validation Profile

So sánh cấu trúc dự kiến từ các đơn vị xử lý Bronze đang chờ với Silver Candidate thực tế được Spark tạo ra.

Đối với hợp đồng Forecast API hiện tại:

`Số hàng dự kiến = Số đơn vị đang chờ xử lý × 48`

In [0]:
validation_result = (
    profile_forecast_validation(
        df=df_silver_candidate,
        pending_ingestion_paths=pending_ingestion_paths,
        expected_rows_per_ingestion=48,
    )
)

validation_result

### 6.2. Transformation Validation Assertion

Ngăn việc lưu Silver khi kết quả chuyển đổi chưa đầy đủ hoặc không khớp với tập Bronze đang chờ xử lý.

Chỉ có Silver Candidate được đối chiếu đầy đủ mới có thể tiếp tục ghi lên Silver.

In [0]:
assert_forecast_validation(
    validation_result
)

print("Forecast Transformation Validation PASS")

## 7. Silver Write

Lưu trữ bộ dữ liệu `weather_forecast_hourly` đã được xác thực vào lớp Silver.

Chỉ có một ứng viên Silver đã vượt qua cả hai bước:

- Chất lượng dữ liệu
- Xác thực biến đổi

mới được phép ghi.

Bộ dữ liệu Silver được lưu dưới định dạng Delta để các lần chạy pipeline sau này có thể đọc an toàn các giá trị `ingestion_id` đã xử lý trước đó và chỉ thêm các đơn vị ingestion mới.

### 7.1. Prepare Final Silver Output

Xóa các cột development và lineage-debugging không thuộc về schema Silver chính thức `weather_forecast_hourly`.

DataFrame kết quả đại diện cho tập dữ liệu cuối cùng sẽ được lưu vào lớp Silver.

In [0]:
df_silver_output = (
    prepare_forecast_silver_output(
        df_silver_candidate
    )
)

In [0]:
display(df_silver_output.limit(20))

In [0]:
display(df_silver_output.limit(20))

In [0]:
df_silver_output.printSchema()

### 7.2. Write Silver Delta

Gắn các bản ghi Dự báo đã được xác thực vào tập dữ liệu Silver Delta.

Lô hiện tại chỉ chứa các đơn vị Bronze đang chờ xử lý mà trước đó chưa được lưu trữ trong Silver.

Các ghi giao dịch Delta ngăn các cập nhật bảng bị cam kết một phần hiển thị như dữ liệu Silver hợp lệ.

In [0]:
write_forecast_silver(
    df=df_silver_output,
    silver_path=SILVER_FORECAST_PATH,
)

print("Weather Forecast Silver write completed.")

### 7.3. Read-back Validation

Đọc lại bộ dữ liệu Silver Delta đã lưu từ ADLS và kiểm tra xem các đơn vị nhập dữ liệu đang chờ hiện tại đã được lưu thành công hay chưa.

Điều này xác nhận rằng Silver Candidate đã được xác thực bây giờ là dữ liệu Silver bền vững, chứ không chỉ là trạng thái tạm thời của Spark.

In [0]:
df_silver_persisted = (
    spark.read
    .format("delta")
    .load(SILVER_FORECAST_PATH)
)

In [0]:
display(
    df_silver_persisted
    .orderBy(
        "ingestion_id",
        "forecast_time"
    )
    .limit(20)
)

In [0]:
print(
    "Silver total rows:",
    df_silver_persisted.count()
)

print(
    "Silver distinct ingestions:",
    df_silver_persisted
    .select("ingestion_id")
    .distinct()
    .count()
)

### 7.4. Verify Current Batch Persistence

Xác nhận rằng mọi đơn vị nhập liệu được xử lý bởi lần chạy pipeline hiện tại bây giờ đều tồn tại trong tập dữ liệu Silver Delta đã lưu trữ.

Kết quả thành công sẽ xác nhận rằng batch hiện tại đã được cam kết bền vững.

In [0]:
pending_ingestion_ids = {
    extract_ingestion_id_from_path(path)
    for path in pending_ingestion_paths
}

persisted_ingestion_ids = {
    row["ingestion_id"]
    for row in (
        df_silver_persisted
        .select("ingestion_id")
        .distinct()
        .collect()
    )
}

missing_after_write = (
    pending_ingestion_ids
    - persisted_ingestion_ids
)

if missing_after_write:
    raise ValueError(
        "Silver write verification FAILED. "
        f"Missing ingestion IDs: {missing_after_write}"
    )

print("Silver read-back validation PASS")